# Wykład 4 — Kiedy przychodzi czas na podjęcie decyzji?
### Wyrażenia Warunkowe

## W4.0 — Od wartości do wyboru ścieżki wykonania

W W2 obliczaliśmy wartości i poznaliśmy `bool`; w W3 odczytywaliśmy dane i korzystaliśmy z prostego `if` jako zabezpieczenia.
Przejdziemy dalej i zapytamy: **która instrukcja zostanie wykonana, a której program w ogóle nie odwiedzi?**

**Cel:** przewidzieć wartość warunku, rzeczywistą kolejność ewaluacji i wybraną drogę przez `if`/`else`/`?:`/`switch`.

### Wyrażenia logiczne — pytania z odpowiedzią „tak” albo „nie”

Wartość wyrażenia `x < 0` odpowiada na pytanie „czy `x` jest ujemne?”. W C wynik porównania to `int`: **1** (prawda) albo **0** (fałsz).

| Wyrażenie dla `x = -3` | Wynik |
|:--|--:|
| `x < 0` | 1 |
| `x == 0` | 0 |
| `x != 0` | 1 |

**Wyrażenie logiczne oblicza odpowiedź. `if` wykorzystuje ją do wyboru instrukcji.** Co zmieni się po ustawieniu `x = 3`?

### Pierwsze doświadczenie: ta sama liczba, różny przebieg

Schemat składni (fragment, nie samodzielny program):
```c
if (warunek)
{
    /* instrukcje */
}
```


Przewidź, co zostanie wypisane dla `x = -3`, a potem dla `x = 3`. W tym doświadczeniu używamy już znanego z W3 prostego `if`, bez wyjaśniania całej składni po raz drugi.

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -3;
    printf("Przed warunkiem\n");
    if (x < 0)
    {
        printf("x jest ujemne\n");
    }
    printf("Po warunku\n");
    return 0;
}

### Co zaobserwowaliśmy?

Warunek może zmienić **zbiór wykonanych instrukcji**. Niezależnie od wybranej drogi program kontynuuje działanie za blokiem `if`. Zamień `x = -3` na `x = 3` i przewidź wynik przed ponownym uruchomieniem.

### Jedna instrukcja czy cały blok?

Składnia `if (warunek) instrukcja;` obejmuje **jedną instrukcję**, niezależnie od liczby linii i wcięć. Instrukcję można rozbić na kilka linii. Blok `{ ... }` jest instrukcją złożoną: dopiero klamry grupują kilka instrukcji pod jednym warunkiem.

Porównaj **dwa osobne programy**. Oba startują od `x = -3`. Przewidź wydruk, następnie w obu zmień tylko znak na `x = 3`. Wcięcia pomagają czytać; nie ustalają granic `if`.


In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -3;
    if (x > 0)
        printf(
            "A\n");
    puts("B");
    puts("Koniec");
    return 0;
}


In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -3;
    if (x > 0) {
        printf(
            "A\n");
        puts("B");
    }
    puts("Koniec");
    return 0;
}


Dla `x = -3` pierwszy program drukuje `B` i `Koniec`, drugi tylko `Koniec`. Dla `x = 3` oba drukują `A`, `B`, `Koniec`. Wywołanie `printf` rozpisane na dwie linie nadal jest jedną instrukcją.

### Przypadkowy średnik: pusta instrukcja — antywzorzec

W `if (warunek);` średnik jest **pustą instrukcją**, którą steruje `if`. Następujący po niej blok wykona się niezależnie od warunku. To poprawne składniowo C, ale zwykle błąd intencji. **Przewidź:** dlaczego dla ujemnego `x` też zobaczysz komunikat? Kompilator z ostrzeżeniami powinien pomóc wykryć pułapkę.


In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -3;
    if (x > 0); /* ANTYWZORZEC: pusta instrukcja */
    {
        puts("Blok wykonany");
    }
    return 0;
}


### Zasięg nazwy i czas życia obiektu to różne pojęcia

**Zasięg (scope)** mówi, gdzie w kodzie nazwa oznacza daną zmienną. Deklaracja `int b` w bloku udostępnia nazwę `b` od deklaracji do końca tego bloku. Blok `{...}` tworzy zasięg **także bez `if`**.

**Czas życia** mówi, kiedy obiekt istnieje podczas wykonywania programu. Dla pokazanych zwykłych zmiennych lokalnych o automatycznym czasie przechowywania obiekt istnieje od wejścia do swojego bloku do wyjścia z niego. Inicjalizacja zachodzi, gdy wykonanie dochodzi do deklaracji. Przy kolejnym wykonaniu bloku powstaje nowy obiekt; nie zachowuje poprzedniej wartości. Inne czasy przechowywania i tablice o zmiennej długości poznamy później.

**Przesłanianie:** lokalna `a` zasłania nazwę zewnętrznej `a`, ale zewnętrzny obiekt nadal istnieje. Po zamknięciu bloku znów odwołujemy się do zewnętrznego obiektu.


In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 10;
    {
        int b = 20;
        int a = 30;
        printf("w bloku a=%d b=%d\n", a, b);
    }
    printf("po bloku a=%d\n", a);
    /* printf("%d\n", b); */ /* b poza zasiegiem */
    return 0;
}


**Wynik:** w bloku `a=30 b=20`, po bloku `a=10`. Zewnętrzna `a` nie została zmieniona. **Zmień:** usuń `int` przed wewnętrznym `a = 30`; będzie to przypisanie do zewnętrznej zmiennej, nie nowa deklaracja.

**Celowy błąd kompilacji — EXPECTED-FAIL**, osobna próba, nie poprawny program:

```c
#include <stdio.h>
int main(void)
{
    { int b = 20; printf("%d\n", b); }
    printf("%d\n", b); /* nazwa b poza zasiegiem */
    return 0;
}
```

Oczekujemy diagnostyki o niezadeklarowanej `b`. Dopisanie klamer po `if` nie jest jedynym sposobem utworzenia bloku. W W5 zmienna zadeklarowana w nagłówku `for` będzie widoczna w pętli, lecz nie za nią.


## W4.1 — Co właściwie znaczy „warunek”?

Z W2.9: porównania dają `int` równy 0 albo 1. W warunkach C **zero oznacza fałsz**, a **każda wartość niezerowa — prawdę**.

Wyrażenie oblicza wartość; instrukcja `if` wykorzystuje ją do wyboru ścieżki wykonania.

## W4.2 — Łączymy warunki
### Logiczna suma i iloczyn

**Problem:** czy `x` należy do przedziału `[0, 10]`?

W C konieczność spełnienia na raz więcej niż jednego warunku (iloczyn logiczny) zapisujemy, używając `&&`: `(x >= 0) && (x <= 10)`.

**Przewidź** wyniki dla `x = -1, 0, 7, 10, 11`;

**Pułapka:** `0 <= x <= 10` nie sprawdza przedziału. W C oznacza to `(0 <= x) <= 10`.

Poeksperymentujmy na poniższym przykładzie:

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 7;
    int w_przedziale = (x >= 0) && (x <= 10);
    printf("x=%d, w_przedziale=%d\n", x, w_przedziale);
    return 0;
}

### `&&`, `||`, `!` — tabela prawdy: operatory logiczne AND, OR i NOT

| p | q | \&\& | \|\| | !p |
|:-:|:-:|:--:|:--:|:--:|
| 0 | 0 | 0 | 0 | 1 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 0 |
| 1 | 1 | 1 | 1 | 0 |

`&&` wymaga obu warunków; `||` co najmniej jednego; `!` neguje. Wyniki tych operatorów mają typ `int` i wynoszą 0 albo 1. **Każdy niezerowy operand to prawda** — także ujemny.

**Sprawdź w C:** czy `-2 && 7` daje `1`, a może `7`?

In [ ]:
#include <stdio.h>

void wiersz(int p, int q)
{
    printf("%d %d  %d   %d   %d\n", p, q, p && q, p || q, !p);
}

int main(void)
{
    puts("p q AND OR NOT_p");
    wiersz(0, 0);
    wiersz(0, 1);
    wiersz(1, 0);
    wiersz(1, 1);
    printf("-2 && 7 = %d, -2 || 0 = %d, !(-2) = %d\n", -2 && 7, -2 || 0, !(-2));
    return 0;
}

### Negacja warunku — prawa de Morgana

`!(p && q)` jest równoważne `(!p || !q)`.  
`!(p || q)` jest równoważne `(!p && !q)`.

Warunek `!(x >= 0 && x <= 10)` jest równoważny `(x < 0) || (x > 10)`.

**Sprawdź** dla `x = -1, 0, 10, 11`.

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 11;
    int w = (x >= 0) && (x <= 10);
    int poza1 = !w;
    int poza2 = (x < 0) || (x > 10);
    printf("x=%d, w=%d, !w=%d, poza=%d\n", x, w, poza1, poza2);
    return 0;
}

## W4.3 — Priorytet i kolejność obliczania

Wśród operatorów logicznych **priorytet (wiązanie wyrażenia)** jest następujący: `!` → `&&` → `||`.

`p || q && r` oznacza `p || (q && r)`, a nie `(p || q) && r`.

**Przewidź** wyniki dla `p = 1, q = 0, r = 0`.

![G1 — drzewo wyrażenia](w4_g1_expression.svg)

In [ ]:
#include <stdio.h>

int main(void)
{
    int p = 1, q = 0, r = 0;
    printf("p || (q && r) = %d\n", p || (q && r));
    printf("(p || q) && r = %d\n", (p || q) && r);
    return 0;
}

### Short-circuit: prawa strona nie zawsze jest obliczana

Dla `p && q`: gdy `p` jest fałszem, `q` zostaje pominięte.  
Dla `p || q`: gdy `p` jest prawdą, `q` zostaje pominięte.

`&&` i `||` obliczają lewy operand przed prawym. **Priorytet operatorów to nie kolejność ewaluacji.**

Funkcja `prawy_operand()` wypisuje komunikat, jeżeli zostanie wywołana. **Przewidź, ile razy.**
![G2 — short-circuit](w4_g2_short_circuit.svg)

In [ ]:
#include <stdio.h>

int prawy_operand(int wartosc)
{
    puts("Obliczam prawy operand");
    return wartosc;
}

int main(void)
{
    int p = 0;
    int wynik = p && prawy_operand(1);
    printf("0 && P: %d\n", wynik);

    p = 1;
    wynik = p && prawy_operand(1);
    printf("1 && P: %d\n", wynik);

    p = 1;
    wynik = p || prawy_operand(0);
    printf("1 || P: %d\n", wynik);

    p = 0;
    wynik = p || prawy_operand(0);
    printf("0 || P: %d\n", wynik);
    return 0;
}

### Zastosowanie: możemy uniknąć dzielenia przez zero

`(mianownik != 0) && (licznik / mianownik > 2)` nie wykonuje dzielenia, gdy mianownik jest zerem.

**Sprawdź** dla `mianownik = 0, 2, 5`. Zastąpienie `&&` przez `&` nie jest bezpieczne, bo `&` znaczy coś zupełnie innego i nie ma short-circuit.

In [ ]:
#include <stdio.h>

int main(void)
{
    int licznik = 10, mianownik = 0;
    int wynik = (mianownik != 0) && (licznik / mianownik > 2);
    printf("mianownik=%d, wynik=%d\n", mianownik, wynik);
    return 0;
}

### Pytania kontrolne

1. Oblicz `!(3 < 5 || 4 > 8)`. Czy oba porównania zostaną wykonane?
2. Czy dla `p = 1` wyrażenie `p || prawy_operand(0)` wywoła funkcję?
3. Zapisz warunek: `x` jest poza przedziałem `[-2, 3)`. Sprawdź wartości brzegowe.

**Dalej:** `if` i `else` wykorzystają wyniki tych wyrażeń do wyboru gałęzi.

## W4.4 — Jedno pytanie czy wiele niezależnych decyzji?
### `else` i `else if`

**Problem:** Co jeżeli mamy do sprawdzenia kilka warunków, ale kolejne chcemy weryfikować tylko, gdy poprzednie nie zostaną spełnione.
Czy trzy instrukcje `if` zrealizują tę potrzebę? Czy może potrzebny jest dodatkowy konstrukt?
Czy trzy razy `if` zachowają się jak jedna konstrukcja `if / else if / else` dla `a == b`?

![G3 — niezależne `if` i łańcuch](w4_g3_branching.svg)

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 2;
    if (x > 0) {
        puts("dodatni");
        puts("wewnatrz bloku");
    } else {
        puts("niedodatni");
    }
    puts("po decyzji");
    return 0;
}

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 7;
    if (x > 0) puts("A");
    if (x < 10) puts("B");
    if (x == 7) puts("C");
    puts("---");
    if (x < 0) puts("ujemne");
    else if (x == 0) puts("zero");
    else puts("dodatnie");
    return 0;
}

### Do którego `if` należy `else`?

Bez dodatkowych klamer `else` wiąże się z **najbliższym poprzedzającym `if`, który nie ma jeszcze swojego `else`**. Wcięcie nie zmienia tej reguły.

Fragment do prześledzenia (antywzorzec, może wywołać ostrzeżenie):
```c
if (a)
    if (b)
        puts("oba prawdziwe");
    else
        puts("a prawdziwe, b falszywe");
```

Dla `a=0` nie wykona się żadna gałąź; ten `else` nie należy do zewnętrznego `if`. **Dobra praktyka:** stosuj klamry, zwłaszcza w zagnieżdżeniach. Poniższy pełny program pokazuje jawnie oba poziomy.


### Prawdziwy warunek, do którego program nie dotarł

**Przewidź:** dla `x = -2` prawdziwe są zarówno `x < 10`, jak i `x == -2`. Czy poniższy łańcuch wypisze B i C?

Nie: pierwsza prawdziwa gałąź kończy wybór w całym łańcuchu. Wynik to B i separator. Trzeci warunek nie zostanie sprawdzony. Co więcej, gałąź C jest tutaj nieosiągalna: każda liczba równa −2 spełnia wcześniejsze `x < 10`. To przykład niepoprawnej kolejności warunków, mimo poprawnej składni.

`else if` to `else` połączone z kolejnym `if`; przeniesienie słowa `if` do następnego wiersza nie zmieni tej zasady. **Zmień:** przesuń szczególny przypadek `x == -2` przed ogólny `x < 10`. Następnie porównaj z trzema niezależnymi `if`.


In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -2;
    if (x > 0)
        puts("A");
    else if (x < 10)
        puts("B");
    else if (x == -2)
        puts("C");
    puts("---");
    return 0;
}


In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 1, b = 0;
    if (a) {
        if (b) puts("wewnetrzny if");
        else puts("else nalezy do wewnetrznego if");
    } else {
        puts("zewnetrzny else");
    }
    return 0;
}

A co będzie oznaczać `if (a = 7)`? To poprawne składniowo wyrażenie przypisania w warunku, nie porównanie. Może, ale nie musi skończyć się ostrzeżeniem i spowodować nieoczekiwane skutki uboczne.

In [ ]:
#include <stdio.h>

int main(void)
{
    int a = 0;
    if ((a = 7)) puts("warunek prawdziwy");
    printf("a=%d\n", a);
    return 0;
}

**Uwaga:** `if (a = 7)` wykonuje przypisanie i testuje jego wynik; zapis z dodatkowymi nawiasami jest celowy. Kompilator może ostrzegać o przypisaniu w warunku. To nie jest `a == 7`.

## W4.5 — Gdy wybieramy wartość, a nie blok instrukcji

**Problem:** jak znaleźć większą z dwóch liczb, korzystając z `?:`, i kiedy czytelniejszy jest `if/else`?

Zestawiamy operator warunkowy `?:` z instrukcjami rozgałęziającymi. `?:` jest wyrażeniem — daje wartość; `if` steruje wykonaniem instrukcji.

`wynik = warunek ? wartosc_gdy_prawda : wartosc_gdy_falsz`

In [ ]:
#include <stdio.h>

int main()
{
    int a = 10;
    int b = 5;
    int c = 7;
    
    int d;
    
    if(b > c)
        d = a + b;
    else
        d = a + c;
    printf("%d\n", d);
}

In [ ]:
#include <stdio.h>

int main()
{
    int a = 10;
    int b = 5;
    int c = 7;
    
    int d = b > c ? a + b : a + c;
    printf("%d\n", d);
}

## W4.6 — Wybór przypadku: switch

**Problem:** jak na podstawie dyskretnego kodu decyzji wybrać jedną z kilku akcji?

W `switch` używamy stałych etykiet `case`, `break`, `default` i świadomego fall-through; `switch` nie oznacza dowolnego warunku logicznego.

In [ ]:
#include <stdio.h>

int main()
{
    int a = 3;
    
    switch( a )
    {
        case 1:
            printf("This is the first case\n");
        break;
        
        case 3:
            printf("This is the second case\n");
        break;
        
        case 5:
            printf("This is the third case\n");
        break;
        
        default:
            printf("This is the default\n");
    }
}

In [ ]:
#include <stdio.h>

int main(){
  int a;
  //scanf("%d", &a);
  
  a = 6;
  int b = 5;
  
  switch(a-b)
  {
    case 1: // the value here is what needs to be evaluated in the switch statment
      printf("This is the first case\n");
      //possibly many lines
      break;
      //some more instructions?
      
    case 3:
      printf("This is the second case\n");
      break;
    
    case 4:
      printf("This is the 4th case\n");
      break;
          
    default:
      printf("Your choiche is unrecognized!!\n");
  }
}

## Przed finałem: funkcje i zasięg zmiennych

Z W1–W2 wiemy, że funkcja może przyjmować argumenty oraz zwracać wartość. **Argumenty w C są przekazywane przez wartość:** funkcja modyfikuje swoje lokalne kopie, nie zmienne wywołującego.

**Przewidź**, czy po wywołaniu `zwieksz_kopie(x)` zmieni się `x` w `main`.

Przypomnienie: w C **nie ma przeciążania nazw funkcji** na podstawie argumentów. Prototyp zapowiada funkcję, a definicja zawiera jej kod.
Parametr w **definicji funkcji** jest lokalną nazwą widoczną od deklaracji do końca jej ciała. Parametr `x` i `x` w `main` oznaczają różne obiekty; nazwa parametru nie jest dostępna w innych funkcjach. Typowy parametr wartościowy istnieje podczas danego wywołania.


In [ ]:
#include <stdio.h>

static void zwieksz_kopie(int x) { x += 1; printf("lokalna=%d\n", x); }

int main(void)
{
    int x = 4;
    zwieksz_kopie(x);
    printf("po wywolaniu=%d\n", x);
    return 0;
}

### Zmienne globalne i przesłanianie nazw

Zmienna zadeklarowana **poza funkcjami** ma zasięg pliku. Funkcje mogą korzystać z niej i zmieniać jej wartość, o ile jej deklaracja jest widoczna.

Zmienna lokalna o tej samej nazwie **przesłania** globalną w swoim bloku. Oba obiekty nadal istnieją.

**Przewidź**: dlaczego funkcja wypisze `15`, a `main` wypisze `100`?

In [ ]:
#include <stdio.h>

int x = 10;

static void zmien(void)
{
    x += 5;
    printf("funkcja: x=%d\n", x);
}

int main(void)
{
    int x = 100;
    zmien();
    printf("main: x=%d\n", x);
    return 0;
}

## W4.7 — To samo zadanie, dwa sposoby organizacji programu

Rozwiązujemy `ax² + bx + c = 0` w liczbach rzeczywistych. **A i B mają tę samą matematykę, dane, komunikaty i politykę przypadków.** Niepoprawny odczyt lub nieskończone/NaN współczynniki kończą program statusem 1. Poprawnie rozpoznany przypadek matematyczny (także brak rozwiązań) kończy go statusem 0.

Dla `a != 0` obliczamy `delta = b*b - 4*a*c` i rozróżniamy jej znak. Dla `a == 0` rozpatrujemy równanie liniowe: `b != 0` daje `x=-c/b`; `b==0, c!=0` oznacza sprzeczność; `b==0, c==0` to tożsamość.

`isfinite(x)` z `<math.h>` daje prawdę dla skończonej liczby; `!isfinite(x)` odrzuca NaN i nieskończoność. `sqrt` oraz `-lm` znamy z W3. Zakładamy umiarkowane współczynniki i reprezentowalne wyniki działań. Jest to ćwiczenie rozgałęzień: dla danych blisko delty zero porównanie `== 0.0` jest wrażliwe na zaokrąglenia (W2), a zwykły wzór może tracić cyfry przy odejmowaniu bliskich liczb. Nie traktuj go jako uniwersalnego solvera numerycznego.

![Drzewo decyzji równania](w4_g4_quadratic.svg)

**Oba poniższe programy są terminalowe.** Zapisz każdy osobno jako `/tmp/w4_a.c` i `/tmp/w4_b.c`. Nie uruchamiaj ich jako komórek kodu C w Jupyterze, bo `scanf` oczekuje wejścia. Deterministyczne próby niżej podają je bez klawiatury.


### A — wszystkie etapy w `main`

Przewidź drogę dla `1 -3 2` i `0 2 -4`. Najpierw odczyt i kontrola, potem wybór gałęzi, obliczenia i wydruk. Nie ma własnych funkcji.

```c
#include <math.h>
#include <stdio.h>

int main(void)
{
    double a, b, c;
    if (scanf("%lf %lf %lf", &a, &b, &c) != 3) {
        puts("Blad danych");
        return 1;
    }
    if (!isfinite(a) || !isfinite(b) || !isfinite(c)) {
        puts("Blad danych");
        return 1;
    }
    if (a == 0.0) {
        if (b == 0.0) {
            if (c == 0.0) puts("Nieskonczenie wiele rozwiazan");
            else puts("Brak rozwiazan");
        } else {
            printf("Liniowe x=%.9g\n", -c / b);
        }
    } else {
        double delta = b * b - 4.0 * a * c;
        if (delta < 0.0) {
            puts("Brak pierwiastkow rzeczywistych");
        } else if (delta == 0.0) {
            printf("Jeden x=%.9g\n", -b / (2.0 * a));
        } else {
            double pierw = sqrt(delta);
            printf("Dwa x1=%.9g x2=%.9g\n",
                   (-b + pierw) / (2.0 * a), (-b - pierw) / (2.0 * a));
        }
    }
    return 0;
}
```


### B — podział na funkcje i procedury

Wczytanie trzech współczynników z kontrolą pozostaje w `main`. Dzięki temu nie wprowadzamy jeszcze funkcji zapisującej kilka wyników przez wskaźniki. Pozostałe etapy mają nazwy: obliczenie delty → decyzja o liczbie pierwiastków → obliczenie i wydruk; przypadek liniowy ma osobną procedurę.

Prototypy przed `main` zapowiadają typ wyniku i parametrów. Definicje są niżej. Funkcja `oblicz_delte` zwraca liczbę, `liczba_pierwiastkow` zwraca kod 0/1/2, a procedury `void` wypisują wyniki. `return` w pomocniczej funkcji kończy tylko jej wywołanie; `return` w `main` ustala status programu.

```c
#include <math.h>
#include <stdio.h>

double oblicz_delte(double a, double b, double c);
int liczba_pierwiastkow(double delta);
void wypisz_kwadrat(double a, double b, double delta, int ile);
void wypisz_liniowe(double b, double c);

int main(void)
{
    double a, b, c;
    /* Odczyt pozostaje tu: nie potrzebujemy funkcji zwracajacej trzy liczby. */
    if (scanf("%lf %lf %lf", &a, &b, &c) != 3) {
        puts("Blad danych");
        return 1;
    }
    if (!isfinite(a) || !isfinite(b) || !isfinite(c)) {
        puts("Blad danych");
        return 1;
    }
    if (a == 0.0) {
        wypisz_liniowe(b, c);
    } else {
        double delta = oblicz_delte(a, b, c);
        int ile = liczba_pierwiastkow(delta);
        wypisz_kwadrat(a, b, delta, ile);
    }
    return 0;
}

double oblicz_delte(double a, double b, double c)
{
    return b * b - 4.0 * a * c;
}

int liczba_pierwiastkow(double delta)
{
    if (delta < 0.0) return 0;
    else if (delta == 0.0) return 1;
    else return 2;
}

void wypisz_kwadrat(double a, double b, double delta, int ile)
{
    if (ile == 0) {
        puts("Brak pierwiastkow rzeczywistych");
    } else if (ile == 1) {
        printf("Jeden x=%.9g\n", -b / (2.0 * a));
    } else {
        double pierw = sqrt(delta);
        printf("Dwa x1=%.9g x2=%.9g\n",
               (-b + pierw) / (2.0 * a), (-b - pierw) / (2.0 * a));
    }
}

void wypisz_liniowe(double b, double c)
{
    if (b == 0.0) {
        if (c == 0.0) puts("Nieskonczenie wiele rozwiazan");
        else puts("Brak rozwiazan");
    } else {
        printf("Liniowe x=%.9g\n", -c / b);
    }
}
```


### Sprawdź, czy oba programy robią to samo

```bash
gcc -std=c11 -Wall -Wextra -pedantic /tmp/w4_a.c -o /tmp/w4_a -lm
gcc -std=c11 -Wall -Wextra -pedantic /tmp/w4_b.c -o /tmp/w4_b -lm
printf '1 -3 2\n' | /tmp/w4_a
printf '1 -3 2\n' | /tmp/w4_b
printf '0 2 -4\n' | /tmp/w4_a
printf '0 2 -4\n' | /tmp/w4_b
printf 'kot\n' | /tmp/w4_b
/tmp/w4_b < /dev/null
```

| Dane w obu wersjach | Oczekiwana klasa i wartości | Status |
|:--|:--|--:|
| `1 -3 2` | Dwa pierwiastki: 2 i 1 | 0 |
| `1 2 1` | Jeden: −1 | 0 |
| `1 0 1` | Brak pierwiastków rzeczywistych | 0 |
| `0 2 -4` | Liniowe: 2 | 0 |
| `0 0 1` | Brak rozwiązań | 0 |
| `0 0 0` | Nieskończenie wiele rozwiązań | 0 |
| `kot`, niepełne dane lub puste wejście | Błąd danych | 1 |

**Co zyskaliśmy w B?** Nazwy etapów pomagają zrozumieć `main`. Deltę i klasyfikację można testować osobno; poprawa jednego etapu ma konkretne miejsce, a funkcję obliczającą deltę można ponownie wykorzystać. Nie zmieniliśmy równania. Parametry są kopiami wartości; nie ma globalnych współczynników ani zastępowania odczytanych danych stałymi.

`wypisz_kwadrat` zakłada `a != 0` i `ile == liczba_pierwiastkow(delta)`; `main` zapewnia te warunki. Oddzielne etapy mają własne kontrakty, a nie dowolne zestawy argumentów.

**Nawiązanie do W1/W2:** prototyp to deklaracja, definicja zawiera ciało, argumenty są przekazywane przez wartość. Funkcje można też zdefiniować w osobnych plikach `.c`, skompilować do `.o` i połączyć linkerem; wspólny nagłówek `.h` dostarcza prototypów. To te same etapy kompilacji co na W1. Sam prototyp bez dołączonej definicji nie wystarczy do zlinkowania programu. Użyte wcześniej `static` przy pomocniczej funkcji ogranicza jej widoczność dla linkera do danego pliku; w tych samodzielnych programach nie jest potrzebne.

**Zmień:** sposób wydruku pierwiastków w B, pozostawiając obliczenie delty bez zmian. Następnie sprawdź odczyt `1 kot 2`. Czy program może wtedy używać `c`? Nie — kończy się przed obliczeniami.


## W4.8 — Co umiemy? Co dalej?

**Pytania kontrolne:**

1. Czy z wartości warunku można wywnioskować, które operandy obliczono?
2. Ile gałęzi może wykonać jeden łańcuch `if/else if/else`?
3. Czym różni się `?:` od `if`?
4. Co zmienia usunięcie `break` z `switch`?
5. Dlaczego globalna `x` i lokalna `x` mogą mieć inne wartości?

**Dalej:** W5 — pętle i powtarzanie instrukcji. Analiza ścieżek wykonania przyda się również tam.